# Lab 5. QAOA — Finding Good Cuts and Evaluating the Output Distribution

## Introduction

In this lab, <strong>you will divide a graph's vertices into two groups to find a high-scoring cut.</strong> Interpret each measured bitstring as a partition and adjust the circuit angles to increase the average score. After the search, examine both the average score and how often the best answers appear.

As in Lab 4, repeat <strong>state preparation → evaluation → angle update</strong>. Here, a graph-partition score replaces energy, so first define how to represent and evaluate an answer.

A graph consists of vertices representing objects and edges connecting pairs of vertices. In this lab, they are

$$
V=\{0,1,2,3\},
\qquad \mathcal E=\{(0,1),(1,2),(2,3),(3,0),(0,2)\}
$$

Placing the vertices in order around a square gives its four sides and one diagonal. Edges are undirected, so $(3,0)$ and $(0,3)$ are the same connection and are counted only once.

The goal is to <strong>maximize the number of edges connecting different groups</strong>. These are called “cut edges,” but the edges are not physically deleted. Giving each cut edge 1 point and maximizing the total defines the <strong>MaxCut</strong> problem. For example, placing vertices 0 and 1 in one group and 2 and 3 in the other cuts $(1,2),(3,0),(0,2)$, for a score of 3.

Assign one group bit to each vertex to represent a partition as a bitstring. Vertex $i$ corresponds to qubit qi, and strings use <strong>$z_3z_2z_1z_0$ order</strong>: vertex 0 is on the right. In the preceding partition, putting vertices 0 and 1 in group 1 gives `0011`. Partitions with an empty group are also candidates, so there are 16 possible bitstrings. Problem 1 practices reading them and calculating their scores.

Repeated circuit executions can produce different bitstrings. If bitstring $z$ has output probability $P(z)$ and score $C(z)$, the mean score is

$$
\langle C\rangle=\sum_z P(z)C(z)
$$

You will search for circuit angles that increase this mean.

The circuit starts in the uniform superposition $|+\rangle^{\otimes4}$ without initially favoring any partition. A <strong>cost operation</strong> attaches phases according to bitstring scores, and a <strong>mixer operation</strong> combines components. Changing phase alone leaves Z probabilities unchanged, but relative phases can affect how amplitudes add or cancel during the subsequent mixing. Evaluate the actual distribution after connecting both operations.

Alternating score-dependent operations and mixers, with a classical optimizer choosing their angles, is called <strong>QAOA (Quantum Approximate Optimization Algorithm)</strong>. Using cost angle $\gamma$ and mixer angle $\beta$, you will prepare

$$
|\psi(\gamma,\beta)\rangle
=U_B(\beta)U_C(\gamma)|+\rangle^{\otimes4}
$$

Read the operators from right to left: uniform superposition, cost, then mixer. One cost–mixer pair is called <strong>$p=1$</strong>; $p$ is not the total gate count. Problem 2 explains the operation definitions and how to translate their angles into gates.

Not every pair of angles improves the score. The provided optimizer compares evaluations as it changes the angles, minimizing the negative mean score $-\langle C\rangle$. Check the returned angles by preparing the circuit again.

<strong>Distinguish mean score from optimal-solution probability.</strong> The mean includes every answer's score, whereas optimal-solution probability sums the probabilities of all maximum-scoring bitstrings. A higher mean does not necessarily mean optimal answers appear more often. Unlike Lab 4's unique ground state, several bitstrings may be optimal, so one answer's probability is not enough to evaluate performance.

This problem is small enough to enumerate all 16 candidates. Use that enumeration to evaluate circuit outputs; the circuit itself must implement the given QAOA definition for general input angles.

| Problem | What you will investigate |
|---|---|
| <strong>1. Defining scores and comparison references</strong> | Read bitstrings as partitions and build a score function and classical reference table. |
| <strong>2. Translating scores into circuit operations</strong> | Connect the score observable, cost, and mixer to build the circuit. |
| <strong>3. Searching angles and examining circuit roles</strong> | Optimize, then compare circuits with an operation removed or the order reversed. |
| <strong>4. Evaluating the output distribution</strong> | Distinguish mean score from optimal-solution probability, and exact values from sample results. |

Finally, double every edge's score and decide how to adjust the angles to preserve the original output distribution. Examine whether changing score units means obtaining better answers.

---

## How to Use This Notebook

Each section moves from the purpose and symbol definitions to equations, examples, and exercises. Distinguish <strong>a partition's score, the observable representing that score, and the circuit operation changing the state</strong>.

Run cells from top to bottom, including the provided code for reading bitstrings, calculating states and probabilities, optimizing, generating measurement records, and displaying tables. You do not need to analyze every implementation detail; each problem explains the tools it needs.

1. <strong>Write calculations and predictions first.</strong> Identify the graph, score, initial state, operation order, and angles, then predict the result.
2. <strong>Complete and check the TODOs.</strong> Fill in the area between `TODO BEGIN` and `TODO END`, keeping the function names and parameters unchanged. Run the answer cell, followed by its public check cell.
3. <strong>Write your predictions, then run the relevant experiment.</strong> In the student notebook, `RUN_OBSERVATIONS` starts as `False`. After writing your prediction, set it to `True` and run the configuration cell and the relevant experiment cell. For each new question, predict before opening its results. To prevent further experiments from running, set it to `False` and rerun the configuration cell. Public function checks work independently of this setting.
4. <strong>Compare the results with your predictions and explain them.</strong> Identify the partition represented by an output and distinguish one answer from the performance of the full distribution. After editing code, rerun the answer, check, and connected experiment cells. If the circuit changes, recompute earlier optimization results.

| Function you will write | Role |
|---|---|
| `cut_score` | Calculate the partition score of one bitstring. |
| `cost_layer` | Implement the score-dependent phase operation. |
| `qaoa_p1` | Connect initial-state preparation, cost, and mixer. |
| `distribution_metrics` | Calculate mean score and optimal-solution probability from output records. |

The optimizer and objective are provided. Your functions must handle different inputs rather than directly return results for selected bitstrings or angles. For the final application, enter your predictions and proposed angles, then enable the separate `RUN_APPLICATION` setting.

Rotation angles are in <strong>radians</strong>. Statevectors and strings use $q_3q_2q_1q_0$ order; `vertex_bit` can help read vertex bits. Count each undirected edge only once.

The first part uses <strong>exact probabilities and expectation values</strong> of ideal states. Later, prepare fresh states a fixed number of times and record their Z measurement bitstrings. These repetitions are called <strong>shots</strong>. Averaging their scores estimates the mean score; the fraction of optimal records estimates optimal-solution probability. Finite samples fluctuate, so distinguish exact values from estimates. Normalize counts by their total, and treat absent bitstrings as having count 0.

A check status of <strong>Incomplete</strong> means a TODO remains, <strong>Failed</strong> means an execution error or an unmet condition, and <strong>Passed</strong> means the public code check passed. Circuit checks test whether the defined state is prepared; performance evaluation asks how good its output partitions are. These are different questions, so identify the conditions and metrics you compare in your written answers.

<strong>You may use AI to study in this lab.</strong> Use it for concept explanations, equation checks, and debugging, but run suggested code yourself. Explain in your own words, using the results you obtain, why the scoring, circuit construction, and performance evaluation are appropriate.

## Preparing the Environment

This notebook was verified with Python 3.12, Qiskit 2.3.1, NumPy 2.2.6, and SciPy 1.18.1. For local use, install the accompanying `requirements.txt` and select that environment as your Jupyter kernel.

If the packages are missing, run the following command in a new code cell, restart the kernel, and run the notebook from the top.

```python
%pip install qiskit==2.3.1 numpy==2.2.6 scipy==1.18.1
```

Run the provided tools cell below as well. Long code may appear collapsed. Each problem explains the tools and how to use them, so you do not need to analyze their entire implementation before starting.

In [ ]:
import sys
from importlib.metadata import version
print("Python", sys.version.split()[0])
for package in ("qiskit", "numpy", "scipy"):
    print(package, version(package))

### Provided Tools and Experiment Settings

Run the tools cell below. Public checks support learning; written answers must be completed separately. Write your prediction and calculations for each question before viewing its experiment results.

The student notebook starts with `RUN_OBSERVATIONS=False`. After writing your prediction, change the setting below to `True`, run that cell, and run the experiment cell for the question. This setting is not a function TODO. The final application has a separate execution setting.

In [ ]:
'Provided tools: operators, states, sampling, optimization, and learning checks for four-vertex MaxCut.'
import numbers
from html import escape
from types import CodeType, FunctionType, ModuleType
from contextlib import contextmanager
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize
from IPython.display import HTML, display
from qiskit import QuantumCircuit
from qiskit.circuit import Gate, Instruction
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector, state_fidelity

N_VERTICES = 4
EDGES = ((0, 1), (1, 2), (2, 3), (3, 0), (0, 2))
BITSTRINGS = tuple(format(index, '04b') for index in range(16))
ATOL = 1e-9
CHECK_LABELS = ('1b · Partition score', '2b · Cost operation', '2c · QAOA layer', '4b · Distribution metrics')
CHECK_STATUS = {}
LAB_RESULTS = {}
INITIAL_STATE = Statevector(np.ones(16, dtype=complex) / 4)


def show_table(headers, rows):
    def table_text(value):
        if isinstance(value, (float, np.floating)):
            return f'{value:.8f}'
        if isinstance(value, (tuple, list)):
            return '(' + ', '.join(table_text(item) for item in value) + ')'
        return str(value)
    rows = list(rows)
    if any(len(row) != len(headers) for row in rows):
        raise ValueError('Check the number of columns in the table.')
    header = ''.join(f'<th>{escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(table_text(x))}</td>' for x in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def _real(value, label='Return value'):
    if not isinstance(value, numbers.Real) or isinstance(value, (bool, np.bool_)) or not np.isfinite(value):
        raise AssertionError(f'{label}: a finite real number is required.')
    return float(value)


def _scale(value):
    value = _real(value, 'cost_scale')
    if value <= 0:
        raise ValueError('cost_scale must be positive.')
    return value


def _integer_score(value):
    if not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)):
        raise AssertionError('cut_score must return an integer.')
    return int(value)


def vertex_bit(bits, vertex):
    return int(bits[-1 - vertex])


def _pauli_word(letter, vertices):
    word = ['I'] * N_VERTICES
    for vertex in vertices:
        word[N_VERTICES - 1 - vertex] = letter
    return ''.join(word)


def cost_operator(cost_scale=1.0):
    scale = _scale(cost_scale)
    return SparsePauliOp.from_list([('IIII', scale * len(EDGES) / 2)] +
                                  [(_pauli_word('Z', edge), -scale / 2) for edge in EDGES])


MIXER_OPERATOR = SparsePauliOp.from_list([(_pauli_word('X', (q,)), 1.0) for q in range(N_VERTICES)])


def definition_cost_matrix(gamma, cost_scale=1.0):
    diagonal = np.diag(cost_operator(cost_scale).to_matrix()).real
    return np.diag(np.exp(-1j * _real(gamma, 'gamma') * diagonal))


def definition_state(gamma, beta, cost_scale=1.0):
    'Independent reference calculated directly from the public exponential-operator definitions.'
    phase = np.diag(definition_cost_matrix(gamma, cost_scale)) * INITIAL_STATE.data
    mixer = expm(-1j * _real(beta, 'beta') * MIXER_OPERATOR.to_matrix())
    return Statevector(np.einsum('ij,j->i', mixer, phase))


@contextmanager
def _instrument_dependencies(function, replacements):
    'Temporarily instrument functions and aliases on the actual call path, then restore every reference.'
    if not isinstance(function, FunctionType):
        raise AssertionError('Write your answer inside the provided function definition.')
    namespaces, functions, originals = {}, {}, {}
    library_prefixes = ('numpy', 'scipy', 'qiskit', 'IPython', 'importlib', 'contextlib')

    def code_names(code):
        names = set(code.co_names)
        for constant in code.co_consts:
            if isinstance(constant, CodeType):
                names.update(code_names(constant))
        return names

    def dependency(value, name=None):
        key = name if name in replacements else getattr(value, '__name__', None)
        if isinstance(value, FunctionType) and key in replacements:
            originals[id(value)] = (value, replacements[key])
            return True
        return False

    def visit(candidate):
        if not isinstance(candidate, FunctionType) or id(candidate) in functions:
            return
        if dependency(candidate):
            return
        if (candidate.__module__ or '').startswith(library_prefixes):
            return
        functions[id(candidate)] = candidate
        namespace = candidate.__globals__
        namespaces[id(namespace)] = namespace
        for name in replacements:
            dependency(namespace.get(name), name)
        names = code_names(candidate.__code__)
        for name in names:
            value = namespace.get(name)
            if isinstance(value, FunctionType):
                if not dependency(value, name):
                    visit(value)
            elif isinstance(value, ModuleType) and not value.__name__.startswith(library_prefixes):
                module_namespace = vars(value)
                namespaces[id(module_namespace)] = module_namespace
                for attribute in names:
                    target = module_namespace.get(attribute)
                    if not dependency(target, attribute):
                        visit(target)
        for value in (*tuple(candidate.__defaults__ or ()),
                      *tuple((candidate.__kwdefaults__ or {}).values())):
            if not dependency(value):
                visit(value)
        for cell in candidate.__closure__ or ():
            try:
                value = cell.cell_contents
            except ValueError:
                continue
            if not dependency(value):
                visit(value)

    visit(function)
    saved_bindings, saved_defaults, saved_cells = [], [], []
    seen_cells = set()

    def substituted(value):
        match = originals.get(id(value))
        return match[1] if match is not None and value is match[0] else value

    try:
        for namespace in namespaces.values():
            for name, value in tuple(namespace.items()):
                replacement = substituted(value)
                if replacement is not value:
                    saved_bindings.append((namespace, name, value))
                    namespace[name] = replacement
        for candidate in functions.values():
            defaults, keyword_defaults = candidate.__defaults__, candidate.__kwdefaults__
            new_defaults = None if defaults is None else tuple(substituted(value) for value in defaults)
            new_keywords = None if keyword_defaults is None else {
                name: substituted(value) for name, value in keyword_defaults.items()}
            if (defaults is not None and any(a is not b for a, b in zip(defaults, new_defaults))) or (
                    keyword_defaults is not None and any(keyword_defaults[key] is not new_keywords[key] for key in keyword_defaults)):
                saved_defaults.append((candidate, defaults, keyword_defaults))
                candidate.__defaults__, candidate.__kwdefaults__ = new_defaults, new_keywords
            for cell in candidate.__closure__ or ():
                if id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                try:
                    value = cell.cell_contents
                except ValueError:
                    continue
                replacement = substituted(value)
                if replacement is not value:
                    saved_cells.append((cell, value))
                    cell.cell_contents = replacement
        yield
    finally:
        for cell, value in reversed(saved_cells):
            cell.cell_contents = value
        for candidate, defaults, keyword_defaults in reversed(saved_defaults):
            candidate.__defaults__, candidate.__kwdefaults__ = defaults, keyword_defaults
        for namespace, name, value in reversed(saved_bindings):
            namespace[name] = value


def _check_circuit_operations(circuit, allowed=None, active=None):
    'Check allowed primitive gates and user-defined circuits composed only of those gates.'
    forbidden = {'measure', 'reset', 'initialize', 'set_statevector', 'set_density_matrix',
                 'if_else', 'while_loop', 'for_loop', 'switch_case'}
    active = set() if active is None else active
    if id(circuit) in active:
        raise AssertionError('Recursive composite-gate definitions are not allowed.')
    active.add(id(circuit))
    try:
        if circuit.num_clbits:
            raise AssertionError('Do not include classical bits in a state-preparation circuit.')
        for item in circuit.data:
            operation = item.operation
            if operation.name in forbidden or operation.num_clbits:
                raise AssertionError('Use unitary gates without measurement, initialization, reset, or conditional control flow.')
            if allowed is not None and operation.name in allowed:
                continue
            # Expand only circuits wrapped with to_gate/to_instruction. Do not use
            # decompositions of other primitive gates to bypass the allowed-gate condition.
            if operation.base_class in (Gate, Instruction) and operation.definition is not None:
                _check_circuit_operations(operation.definition, allowed, active)
            elif allowed is not None:
                raise AssertionError('Use the primitive gates allowed for this function. Composite gates are checked internally too.')
    finally:
        active.remove(id(circuit))


def circuit_unitary(circuit, kind=None):
    if not isinstance(circuit, QuantumCircuit):
        raise AssertionError('Return a QuantumCircuit.')
    if (circuit.num_qubits, circuit.num_clbits) != (4, 0) or circuit.parameters:
        raise AssertionError('A four-qubit circuit with bound numeric angles and no classical bits is required.')
    allowed = {'cost': {'rzz', 'rz', 'cx', 'barrier'},
               'qaoa': {'h', 'rzz', 'rx', 'rz', 'cx', 'barrier'}}
    _check_circuit_operations(circuit, None if kind is None else allowed[kind])
    matrix = np.asarray(Operator(circuit).data)
    if not np.isfinite(matrix).all() or not np.allclose(np.einsum('ji,jk->ik', matrix.conj(), matrix), np.eye(16), atol=ATOL, rtol=0):
        raise AssertionError('A finite unitary circuit is required.')
    return matrix


def circuit_state(circuit, kind=None):
    return Statevector(circuit_unitary(circuit, kind)[:, 0].copy())


def probabilities_from_circuit(circuit):
    probabilities = np.asarray(circuit_state(circuit).probabilities(), dtype=float)
    probabilities /= probabilities.sum()
    return dict(zip(BITSTRINGS, map(float, probabilities)))


def mean_cut_from_circuit(circuit, cost_scale=1.0):
    return float(np.real(circuit_state(circuit).expectation_value(cost_operator(cost_scale))))


def classical_reference(cost_scale=1.0, score_function=None):
    'Full 16-bitstring reference table. Uses the current cut_score by default; module callers can supply it explicitly.'
    scale = _scale(cost_scale)
    function = score_function if score_function is not None else globals().get('cut_score')
    if not callable(function):
        raise NotImplementedError('Write and run cut_score first.')
    scores = {bits: scale * _integer_score(function(bits)) for bits in BITSTRINGS}
    maximum = max(scores.values())
    optimal = tuple(bits for bits in BITSTRINGS if scores[bits] == maximum)
    return scores, maximum, optimal


def weight_vector(weights):
    if not isinstance(weights, dict) or not weights:
        raise ValueError('A nonempty dictionary of bitstring weights is required.')
    vector = np.zeros(16)
    for bits, value in weights.items():
        if bits not in BITSTRINGS:
            raise ValueError('Bitstrings must contain four bits in q3q2q1q0 order.')
        weight = _real(value, 'Weight')
        if weight < 0:
            raise ValueError('Weights must be nonnegative.')
        vector[int(bits, 2)] = weight
    if vector.sum() <= 0:
        raise ValueError('The total weight must be positive.')
    return vector / vector.sum()


def sample_counts(circuit, shots=1024, seed=None):
    if not isinstance(shots, (int, np.integer)) or isinstance(shots, (bool, np.bool_)) or shots <= 0:
        raise ValueError('shots must be a positive integer.')
    probabilities = weight_vector(probabilities_from_circuit(circuit))
    counts = np.random.default_rng(seed).multinomial(int(shots), probabilities)
    return {bits: int(value) for bits, value in zip(BITSTRINGS, counts) if value}


def optimize_qaoa(objective, initial=(0.1, 0.1)):
    initial = np.array([_real(value, 'Initial angle') for value in initial], dtype=float)
    if initial.shape != (2,):
        raise ValueError('Initial angles must contain gamma and beta.')
    history = []
    def observed(parameters):
        value = _real(objective(np.asarray(parameters).copy()), 'Objective value')
        history.append({'parameters': tuple(map(float, parameters)), 'objective_value': value, 'mean_cut': -value})
        return value
    result = minimize(observed, initial, method='Nelder-Mead',
                      options={'maxiter': 250, 'xatol': 1e-8, 'fatol': 1e-9})
    return {'parameters': tuple(map(float, result.x)), 'initial': tuple(map(float, initial)),
            'objective_value': float(result.fun), 'mean_cut': -float(result.fun),
            'initial_mean': history[0]['mean_cut'], 'nfev': int(result.nfev), 'nit': int(result.nit),
            'success': bool(result.success), 'message': str(result.message), 'history': history}


def _same_operator(actual, expected):
    overlap = np.vdot(expected.ravel(), actual.ravel())
    if abs(overlap) < ATOL:
        return False
    phase = overlap / abs(overlap)
    return bool(np.allclose(actual, phase * expected, atol=ATOL, rtol=0))


def _run_check(label, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[label] = 'Incomplete'
        print(f'{label}: Incomplete — complete the TODO and run again.')
        return False
    except Exception as error:
        CHECK_STATUS[label] = 'Failed'
        print(f'{label}: Failed — {type(error).__name__}: {error}')
        return False
    CHECK_STATUS[label] = 'Passed'
    print(f'{label}: Passed')
    return True


def check_cut_score(function):
    def check():
        diagonal = np.diag(cost_operator().to_matrix()).real
        for bits, expected in zip(BITSTRINGS, diagonal):
            assert _integer_score(function(bits)) == expected, 'Check bit positions and the edge list.'
    return _run_check(CHECK_LABELS[0], check)


def check_cost_layer(function):
    def check():
        for gamma, scale in ((0.0, 1.0), (0.61, 1.0), (-0.47, 0.4), (1.07, 2.0)):
            actual = circuit_unitary(function(gamma, cost_scale=scale), 'cost')
            expected = definition_cost_matrix(gamma, scale)
            assert _same_operator(actual, expected), "Check the full cost operation's relative phases, sign, and scale. A common global phase is allowed."
    return _run_check(CHECK_LABELS[1], check)


def _check_cost_connection(function):
    'Keep the actual answer intact and check use of the circuit returned by cost_layer.'
    if not isinstance(function, FunctionType):
        raise AssertionError('Write the circuit inside the provided function definition.')
    point = (0.37, -0.26, 1.7)
    def neutral_layer(angle):
        layer = QuantumCircuit(4)
        layer.rzz(0.179 * angle, 0, 3)
        layer.rz(-0.413 * angle, 1)
        layer.rzz(0.29 * angle, 1, 2)
        return layer
    probe = neutral_layer(point[0] * point[2])
    calls = []
    def cost(gamma, cost_scale=1.0):
        effective_angle = _real(gamma, 'gamma') * _scale(cost_scale)
        calls.append((gamma, cost_scale))
        return neutral_layer(effective_angle)
    with _instrument_dependencies(function, {'cost_layer': cost}):
        actual = circuit_state(function(point[0], point[1], cost_scale=point[2]), 'qaoa')
        after_cost = np.einsum('ij,j->i', circuit_unitary(probe), INITIAL_STATE.data)
        mixer = expm(-1j * point[1] * MIXER_OPERATOR.to_matrix())
        expected = Statevector(np.einsum('ij,j->i', mixer, after_cost))
        assert calls, 'Call your cost_layer and connect its circuit.'
        assert state_fidelity(actual, expected) >= 1 - ATOL, 'Use the operation returned by cost_layer after initial preparation and before the mixer.'
    return calls



def check_qaoa_p1(function):
    def check():
        for gamma, beta, scale in ((0.0, 0.0, 1.0), (0.38, 0.21, 1.0), (-0.73, 0.42, 2.0), (1.17, -0.28, 0.4)):
            actual = circuit_state(function(gamma, beta, cost_scale=scale), 'qaoa')
            expected = definition_state(gamma, beta, scale)
            assert state_fidelity(actual, expected) >= 1 - ATOL, 'Check the initial state, cost-then-mixer order, angles, and scale.'
        _check_cost_connection(function)
    return _run_check(CHECK_LABELS[2], check)


def check_distribution_metrics(function):
    def check():
        cases = [
            ({'0011': 6, '0000': 2}, 2.25, 0.0),
            ({'0011': 2, '0000': 1, '0101': 3, '1010': 5}, 38/11, 8/11),
            ({'0001': 0.2, '1000': 0.3, '1010': 0.5}, 3.2, 0.5),
            ({bits: 1 for bits in BITSTRINGS}, 2.5, 0.125),
            ({'0101': 0, '0011': 7}, 3.0, 0.0),
        ]
        for weights, base_mean, expected_optimal in cases:
            for scale in (1e-10, 0.4, 1.0, 2.0):
                answer = function(dict(weights), cost_scale=scale)
                assert isinstance(answer, tuple) and len(answer) == 2, 'Return a (mean_cut,p_opt) tuple.'
                mean, optimal = (_real(value) for value in answer)
                assert np.isclose(mean, scale * base_mean, atol=ATOL, rtol=0), 'Normalize by the total weight and apply the score scale.'
                assert np.isclose(optimal, expected_optimal, atol=ATOL, rtol=0), 'Sum weights for every bitstring in the optimal set of the full 16-candidate problem.'
    return _run_check(CHECK_LABELS[3], check)


def run_demo(label, experiment, requires=()):
    LAB_RESULTS.pop(label, None)
    missing = [key for key in requires if CHECK_STATUS.get(key) != 'Passed']
    if missing:
        print(f'[Waiting] {label}: first pass these code checks: {", ".join(missing)}')
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f'[Waiting] {label}: complete the earlier TODOs.')
        return None
    except Exception as error:
        print(f'[Experiment error] {label}: {type(error).__name__}: {error}')
        return None
    LAB_RESULTS[label] = result
    return result

In [ ]:
RUN_OBSERVATIONS = False
print('Run observation experiments:', RUN_OBSERVATIONS)
print('Write your prediction and calculations for the question first, then set this to True and run the observation cell.')

## Problem 1. How Do We Score a Graph Partition?

In this problem, you will <strong>read the partition represented by a bitstring and write a function that calculates its score</strong>. Then enumerate every bitstring to find the maximum score and the complete set of optimal solutions.

These results provide references for evaluating the quantum circuit's outputs. Incorrect bitstring scores would also lead to incorrect conclusions about circuit performance. First connect vertex numbers, bit positions, and edge scores correctly.

---

### A Bitstring Specifies Two Groups

The vertices are $0,1,2,3$, with the following edges.

$$
\mathcal E
=
\{(0,1),(1,2),(2,3),(3,0),(0,2)\}.
$$

Assign a bit $z_i$ to each vertex $i$. Bit 0 means group 0; bit 1 means group 1.

| Vertex bit | Vertex's group |
|---|---|
| $z_i=0$ | Group 0 |
| $z_i=1$ | Group 1 |

Each vertex belongs to exactly one of the two groups. Specifying all bits defines one partition. Candidates include partitions with an empty group.

Neither group label is preferred. Swapping the labels 0 and 1 does not change which vertices belong together.

In a bitstring, swapping labels means flipping every 0 to 1 and every 1 to 0. For example, `0011` and `1100` represent the same partition with exchanged group labels. Remember that they are nevertheless distinct outcomes in measurement records.

---

### Matching String Positions to Vertex Numbers

Qiskit's four-bit strings use

$$
z_3z_2z_1z_0
$$

order. The rightmost bit is vertex 0, and the leftmost is vertex 3.

For example, read `0011` as follows.

| Vertex $i$ | Bit $z_i$ read from the string | Group |
|---:|---:|---|
| 0 | 1 | Group 1 |
| 1 | 1 | Group 1 |
| 2 | 0 | Group 0 |
| 3 | 0 | Group 0 |

The two groups represented by this string are therefore

$$
G_0=\{2,3\},
\qquad
G_1=\{0,1\}
$$

Here, $G_0,G_1$ are the sets of vertices in groups 0 and 1, respectively.

Rather than calculating string positions yourself, you may use the provided tool

```python
vertex_bit(bits, vertex)
```

It reads the bit for `vertex` from `bits` and returns integer 0 or 1. For example, `vertex_bit("0011", 0)` returns 1 and `vertex_bit("0011", 3)` returns 0.

The tool handles string orientation, leaving your score function to focus on <strong>whether the endpoints of each edge belong to the same or different groups</strong>.

---

### An Edge's Score Depends on Its Endpoints' Groups

Assign edge $(i,j)$ score 1 if its endpoints belong to different groups, and 0 if they belong to the same group.

$$
c_{ij}(z)
=
\begin{cases}
1,&z_i\ne z_j,\\
0,&z_i=z_j.
\end{cases}
$$

The total partition score is the sum over the five edges.

$$
C(z)=\sum_{(i,j)\in\mathcal E}c_{ij}(z).
$$

Every edge has weight 1 here, so $C(z)$ is the number of cut edges.

Edges are undirected: $(i,j)$ and $(j,i)$ describe the same connection. Each appears once in the provided list, so inspect each list entry once.

---

### Worked Example: Scoring `0011`

The groups represented by `0011` are

$$
G_0=\{2,3\},
\qquad
G_1=\{0,1\}
$$

Compare the endpoint bits for each edge.

| Edge | Endpoint bits | Same group? | Edge score |
|---|---|---|---:|
| $(0,1)$ | $1,1$ | Yes | 0 |
| $(1,2)$ | $1,0$ | No | 1 |
| $(2,3)$ | $0,0$ | Yes | 0 |
| $(3,0)$ | $0,1$ | No | 1 |
| $(0,2)$ | $1,0$ | No | 1 |

Therefore,

$$
C(0011)=0+1+0+1+1=3
$$



First read each vertex's group, then compare the groups at each edge's endpoints. Use the same rule when evaluating measurement records later.

For `0000` and `1111`, all vertices belong to one group, so every edge has endpoints in the same group. These inputs can check basic behavior of the score function.

---

### Enumerating All Answers to Build a Reference

Four vertices give 16 possible bitstrings. Scoring each one gives the exact maximum

$$
C_{\max}
=
\max_{z\in\{0,1\}^4}C(z)
$$



Collecting all bitstrings with that maximum score gives the optimal-solution set.

$$
\mathcal Z_{\mathrm{opt}}
=
\{z:C(z)=C_{\max}\}.
$$

Do not record only the first bitstring that attains the maximum. Check for all others with the same score. This full set will later determine the probability of an optimal circuit output.

Examining all candidates is called <strong>exhaustive enumeration</strong>. With just 16 candidates, you can calculate them all and compare later experiments against an exact reference.

The enumeration table calls your `cut_score`. An error there could also corrupt the reported maximum and optimal set, so first check the function against hand calculations and the check cell.

---

### Using Uniform Sampling as a Baseline

To judge improvement after optimization, you need a starting reference. Use the <strong>uniform distribution</strong>, which selects all 16 bitstrings with equal probability.

Each bitstring has probability

$$
P_{\mathrm{unif}}(z)=\frac1{16}
$$

The uniform distribution's mean score is therefore

$$
\langle C\rangle_{\mathrm{unif}}
=
\frac1{16}\sum_z C(z)
$$



Optimal-solution probability is the sum of the probabilities of all bitstrings in the optimal set.

$$
P_{\mathrm{opt,unif}}
=
\sum_{z\in\mathcal Z_{\mathrm{opt}}}\frac1{16}
=
\frac{|\mathcal Z_{\mathrm{opt}}|}{16}.
$$

Here, $|\mathcal Z_{\mathrm{opt}}|$ is the number of optimal bitstrings.

These references also describe the initial quantum state used later. Directly measuring $|+\rangle^{\otimes4}$ in Z gives each bitstring probability $1/16$, hence the same mean score and optimal-solution probability.

Thus, the classical score table lets you evaluate the initial output distribution even before writing a circuit.

---

### Your Turn

### (a) Reading Partitions and Calculating Scores

Consider the following two bitstrings.

```text
0001
1000
```

Identify each vertex's group and complete the table.

| Bitstring | Vertices in group 0 | Vertices in group 1 | Cut edges | Score |
|---|---|---|---|---:|
| `0001` | … | … | … | … |
| `1000` | … | … | … | … |

After calculating, explain the following.

1. Both strings contain exactly one 1. Must the two partition scores therefore agree? Explain using the edges incident on the relevant vertices in this graph.
2. How would you interpret these inputs if you incorrectly read the leftmost bit as vertex 0? Explain why these two examples help check bit order.
3. Which two groups does `1110`, the bitwise complement of `0001`, represent? Compare their scores by examining whether each edge's endpoints are in the same or different groups.

Write your hand calculations and reasoning first. After completing the score function in the next question, compare them with the enumeration results.

---

### Answer 1a

Calculations, results, and reasoning:

…

### (b) Writing a Function That Returns a Bitstring's Score

Write the following function.

```python
cut_score(bits)
```

The input `bits` is a valid four-character binary string in `z3z2z1z0` order, containing only 0 and 1.

The edge list is provided as follows.

```python
EDGES = ((0, 1), (1, 2), (2, 3), (3, 0), (0, 2))
```

You may use `vertex_bit(bits, vertex)` to read a vertex's bit.

The function must satisfy these conditions:

- Inspect each edge in `EDGES` once.
- An edge contributes 1 if its endpoint bits differ, and 0 if they agree.
- Return <strong>one integer</strong> equal to the sum over all five edges.
- Use the same mapping between string positions and vertex numbers for every input.

Calculate from the supplied string and edge list rather than hardcoding scores for particular strings.

First test the worked example `0011` and the two inputs from (a). Then run the check cell, which compares your function with an independent score calculation on every possible bitstring.

After the function passes, run the provided enumeration cell. It uses your function to build a 16-bitstring score table and displays the maximum and every bitstring attaining it.

---

In [ ]:
def cut_score(bits):
    # TODO BEGIN — Check whether each edge's endpoints belong to different groups.
    raise NotImplementedError('cut_score')
    # TODO END

In [ ]:
_ = check_cut_score(cut_score)

### When to View the Score Table

The experiment cell for the score table is placed after the prediction space in 1(c). Predict the maximum and a partition first, then run it. Use the same table to check your hand calculations from 1(a).

### Answer 1b

Calculations, results, and reasoning:

…

### (c) Finding the Maximum and the Uniform Baseline

First inspect the edges and predict the maximum possible score and a partition attaining it. Write your prediction and reasoning before viewing the enumeration table from (b).

Use the result to complete this table.

| Item | Result |
|---|---|
| Maximum score $C_{\max}$ | … |
| Optimal-solution set $\mathcal Z_{\mathrm{opt}}$ | … |
| Number of optimal bitstrings $\vert \mathcal Z_{\mathrm{opt}}\vert $ | … |
| Uniform mean score $\langle C\rangle_{\mathrm{unif}}$ | … |
| Uniform optimal-solution probability $P_{\mathrm{opt,unif}}$ | … |

After completing the table, explain all three points below.

<strong>1. Why the score you found is truly maximal</strong>

Besides enumeration, use the graph's structure to explain the maximum. In particular, examine the three edges

$$
(0,1),\qquad(1,2),\qquad(0,2)
$$

Can vertices 0, 1, and 2 be assigned to two groups so that all three edges are cut?

Connect your answer to the possible total over all five edges, and check whether your partition reaches that bound.

<strong>2. Checking the uniform mean in two ways</strong>

First, add the scores of all 16 bitstrings and divide by 16.

Second, examine the possible endpoint bit pairs for one edge:

$$
(0,0),\ (0,1),\ (1,0),\ (1,1)
$$

When the full bitstring is chosen uniformly, find the probability that this edge is cut, then add the mean contributions of all five edges.

Check that the two methods give the same total mean score.

<strong>3. Deciding which bitstrings count toward optimal-solution probability</strong>

If every bit of an optimal bitstring is flipped, is the result also optimal? Explain using group labels and the cut edges.

Even if two strings represent the same partition, explain why calculating optimal-solution probability from measurements requires <strong>summing the probabilities of all optimal strings</strong>.

### Answer 1c

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Results: 1bc

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_1bc():
    table, maximum, optimal = classical_reference(score_function=cut_score)
    rows = []
    for bits in ("0001", "1000", "1110"):
        groups = [tuple(q for q in range(4) if vertex_bit(bits, q) == bit) for bit in (0, 1)]
        cut_edges = tuple(edge for edge in EDGES if vertex_bit(bits, edge[0]) != vertex_bit(bits, edge[1]))
        rows.append([bits, groups[0], groups[1], cut_edges, table[bits]])
    show_table(['Bitstring', 'Group 0', 'Group 1', 'Cut edges', "Student function's score"], rows)
    show_table(['Bitstring', 'Score'], [[bits, score] for bits, score in table.items()])
    uniform_mean = sum(table.values()) / len(table)
    uniform_optimal = len(optimal) / len(table)
    show_table(['Maximum score', 'All optimal bitstrings', 'Uniform mean score', 'Uniform optimal-solution probability'],
               [[maximum, optimal, uniform_mean, uniform_optimal]])
    return {"score_table": table, "maximum": maximum, "optimal_strings": optimal,
            "uniform_mean": uniform_mean, "uniform_p_opt": uniform_optimal}

if RUN_OBSERVATIONS:
    _ = run_demo("1bc", experiment_1bc, (CHECK_LABELS[0],))
else:
    LAB_RESULTS.pop("1bc", None)
    print('[Waiting] Write predictions and calculations for 1(a,c) before viewing the score table.')

## Problem 2. How Do Scores Become Quantum Circuit Operations?

You have calculated bitstring scores and enumerated all partitions to build a reference. Now investigate <strong>how to encode that score information in a quantum circuit</strong>.

First express the classical score function as a quantum observable. Then build a cost operation that assigns score-dependent phases, and connect a mixer so those phase differences can affect output probabilities.

Implement this in two functions. `cost_layer` handles the score-dependent phase operation, while `qaoa_p1` connects initial-state preparation, cost, and mixer into one circuit.

---

### Expressing an Edge Score with Z Observables

An edge $(i,j)$ scores 1 when its endpoint bits differ and 0 when they agree.

$$
c_{ij}(z)
=
\begin{cases}
1,&z_i\ne z_j,\\
0,&z_i=z_j.
\end{cases}
$$

Translate this rule into Z observables. Acting with Z on a computational-basis state $|z_i\rangle$ gives

$$
Z_i|z_i\rangle=(-1)^{z_i}|z_i\rangle
$$

The eigenvalue of $Z_iZ_j$ on the two qubits is therefore

$$
(-1)^{z_i}(-1)^{z_j}
$$



This product is $+1$ for matching bits and $-1$ for different bits. To convert these values to edge scores 0 and 1, use

$$
\widehat C_{ij}
=
\frac{I-Z_iZ_j}{2}
$$



| Relationship between bits | Eigenvalue of $Z_iZ_j$ | Eigenvalue of $\widehat C_{ij}$ |
|---|---:|---:|
| Same | $+1$ | 0 |
| Different | $-1$ | 1 |

Here, $I$ is the identity. Operators act as the identity on other qubits not named by their subscripts.

The full score operator sums the edge terms.

$$
\widehat C
=
\sum_{(i,j)\in\mathcal E}
\frac{I-Z_iZ_j}{2}.
$$

For a computational-basis state $|z\rangle$,

$$
\widehat C|z\rangle=C(z)|z\rangle
$$

Thus, <strong>the classical score function's return value is the score operator's eigenvalue</strong>.

For a general state

$$
|\psi\rangle=\sum_z a_z|z\rangle
$$

we have

$$
\langle\psi|\widehat C|\psi\rangle
=
\sum_z |a_z|^2C(z)
$$

Since Z measurement gives $P(z)=|a_z|^2$, this expectation value equals the mean score defined earlier.

---

### From a Score Observable to a Phase Operation

You now have an observable for calculating scores. The next goal is to encode that score information in the state.

Applying $\widehat C$ directly multiplies each component's amplitude by $C(z)$. This generally fails to preserve normalization, so it cannot directly serve as a unitary gate.

QAOA instead assigns a score-dependent <strong>phase of absolute value 1</strong>.

$$
U_C(\gamma)
=
e^{-i\gamma\widehat C}.
$$

Because each computational-basis state $|z\rangle$ is an eigenstate of $\widehat C$, this operation acts as

$$
U_C(\gamma)|z\rangle
=
e^{-i\gamma C(z)}|z\rangle
$$



Here, $\gamma$ is a real angle. Components with different scores can acquire different phases, but each phase has absolute value 1, preserving normalization.

Applied to a superposition, it gives

$$
U_C(\gamma)\sum_z a_z|z\rangle
=
\sum_z a_z e^{-i\gamma C(z)}|z\rangle
$$

Each component's Z probability remains $|a_z|^2$, while relative phases can change.

This is the cost operation's role: <strong>first encode scores in relative phases, then let the mixer translate those differences into output amplitudes</strong>.

---

### Splitting the Cost into Edge Operations

The total score is a sum of edge scores,

$$
C(z)=\sum_{(i,j)\in\mathcal E}c_{ij}(z)
$$

so the phase assigned to a bitstring factors as

$$
e^{-i\gamma C(z)}
=
\prod_{(i,j)\in\mathcal E}
e^{-i\gamma c_{ij}(z)}
$$



The same factorization holds for operators. Every $\widehat C_{ij}$ contains only I and Z, so the edge terms commute. This remains true for edges sharing a vertex because both act there with Z.

Therefore,

$$
U_C(\gamma)
=
\prod_{(i,j)\in\mathcal E}
e^{-i\gamma\widehat C_{ij}}
$$



Apply the operations for all five edges in sequence. Reordering these cost operations does not change their combined action, but this does not mean you may also freely exchange their order with the mixer.

---

### Implementing One Edge's Phase with $R_{ZZ}$

For one edge $(i,j)$, the required operation is

$$
e^{-i\gamma\widehat C_{ij}}
=
e^{-i\gamma(I-Z_iZ_j)/2}
$$



Matching endpoint bits score 0 and receive phase 1. Different bits score 1 and receive $e^{-i\gamma}$.

| Endpoint bits | Edge score | Required phase |
|---|---:|---|
| 00 | 0 | $1$ |
| 01 | 1 | $e^{-i\gamma}$ |
| 10 | 1 | $e^{-i\gamma}$ |
| 11 | 0 | $1$ |

The gate used to implement this is

$$
R_{ZZ}(\lambda)
=
e^{-i\lambda Z_iZ_j/2}
$$

Since $Z_iZ_j$ has eigenvalue $+1$ on matching bits and $-1$ on different bits, its action is

$$
R_{ZZ}(\lambda)
=
\begin{pmatrix}
e^{-i\lambda/2}&0&0&0\\
0&e^{i\lambda/2}&0&0\\
0&0&e^{i\lambda/2}&0\\
0&0&0&e^{-i\lambda/2}
\end{pmatrix}.
$$

Compare the two expressions to determine the required rotation angle $\lambda$.

One common phase multiplying every component is allowed. Rather than requiring literal equality of all four diagonal entries, check <strong>whether the necessary relative phase is created between matching-bit and different-bit components</strong>.

An edge's common phase also multiplies the full four-qubit state by that same phase. The product of all five common phases is still a global phase, which is allowed when comparing physical states.

---

### The Mixer Combines Amplitudes

Cost alone leaves Z probabilities unchanged. Next apply an operation that combines each qubit's 0 and 1 components.

The mixer is defined by

$$
\widehat B=\sum_{i=0}^{3}X_i,
\qquad
U_B(\beta)=e^{-i\beta\widehat B}
$$



X operators acting on different qubits commute, so

$$
U_B(\beta)
=
\prod_{i=0}^{3}e^{-i\beta X_i}
$$



The single-qubit rotation gate is defined as

$$
R_x(\lambda)
=
e^{-i\lambda X/2}
=
\begin{pmatrix}
\cos(\lambda/2)&-i\sin(\lambda/2)\\
-i\sin(\lambda/2)&\cos(\lambda/2)
\end{pmatrix}
$$

Compare this definition with $e^{-i\beta X_i}$ to choose the angle applied to each qubit.

In this matrix, both input components contribute to the output 0 component, and likewise to the output 1 component. The mixer therefore <strong>combines complex amplitudes unitarily</strong>, rather than simply averaging bitstring probabilities.

Consider a simple one-qubit example.

$$
|\chi_\phi\rangle
=
\frac{|0\rangle+e^{i\phi}|1\rangle}{\sqrt2}.
$$

Its Z probabilities are $1/2$ for both 0 and 1, independent of $\phi$. After $R_x(\lambda)$, however, the amplitude of 0 is

$$
\frac{
\cos(\lambda/2)
-i e^{i\phi}\sin(\lambda/2)
}{\sqrt2}
$$



Amplitudes from two input components have been added into one output component, and their sum includes the relative phase $\phi$. The measurement probability, the squared magnitude of that sum, can therefore depend on $\phi$.

The same principle applies to four-qubit QAOA. Relative phases created by cost can affect output probabilities after the mixer and thus change the mean score. Actual calculation determines which angles work well.

---

### Assembling One Layer from the Initial State

The state of this $p=1$ circuit is

$$
|\psi(\gamma,\beta)\rangle
=
U_B(\beta)U_C(\gamma)|+\rangle^{\otimes4}
$$



Prepare it in this order:

| Step | Action |
|---|---|
| 1 | Start from $\vert 0000\rangle$. |
| 2 | Apply H to every qubit to prepare $\vert +\rangle^{\otimes4}$. |
| 3 | Apply the cost operations for all five edges. |
| 4 | Apply the mixer to all four qubits. |

Operators act on a state from right to left, so $U_BU_C$ means <strong>cost followed by mixer</strong>.

The circuit syntax is:

| Syntax | Role |
|---|---|
| `circuit.h(q)` | Apply H to q |
| `circuit.rzz(angle, i, j)` | Apply $R_{ZZ}$ with the specified angle to i and j |
| `circuit.rx(angle, q)` | Apply $R_x$ with the specified angle to q |
| `circuit.compose(layer, inplace=True)` | Append `layer` to the circuit using the same qubit order |

Here is a one-qubit example of composing circuits.

```python
layer = QuantumCircuit(1)
layer.x(0)

combined = QuantumCircuit(1)
combined.compose(layer, inplace=True)
```

`combined` now contains the X gate from `layer`. Use the same approach to connect your cost circuit to the full QAOA circuit.

---

### Your Turn

### (a) Comparing Scores with Quantum-Operator Actions

Reuse these two bitstrings from Problem 1.

```text
0001
1000
```

Use their scores to complete the table.

| Bitstring $z$ | Classical score $C(z)$ | $\widehat C\vert z\rangle$ | $U_C(\gamma)\vert z\rangle$ |
|---|---:|---|---|
| `0001` | … | … | … |
| `1000` | … | … | … |

Next consider the two-component superposition

$$
|\eta\rangle
=
\frac{|0001\rangle+|1000\rangle}{\sqrt2}
$$



1. Calculate $\widehat C|\eta\rangle$ and $U_C(\gamma)|\eta\rangle$.
2. Compare the changes to each component's coefficient. Which operation preserves normalization?
3. Check whether both bitstrings still have Z probability $1/2$ after cost.
4. Explain whether the two phases always reduce to one global phase or can produce a relative phase that depends on $\gamma$.

Finally, use the provided operator tools to check that the eigenvalues of $\widehat C$ on all 16 computational-basis states match your `cut_score` results.

You do not need to expand the full $16\times16$ matrix by hand. Verify the correspondence: each computational-basis state is an eigenstate, and its eigenvalue is its partition score.

---

### Answer 2a

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Results: 2a

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_2a():
    diagonal = np.diag(cost_operator().to_matrix()).real
    rows = [[bits, cut_score(bits), diagonal[int(bits, 2)]] for bits in BITSTRINGS]
    show_table(['Bitstring', 'Student classical score', 'Score-operator eigenvalue'], rows)
    gamma = 0.37
    initial = np.zeros(16, dtype=complex)
    initial[int("0001", 2)] = 1 / np.sqrt(2)
    initial[int("1000", 2)] = 1 / np.sqrt(2)
    observed = diagonal * initial
    phased = np.einsum("ij,j->i", definition_cost_matrix(gamma), initial)
    show_table(['Bitstring', 'Original amplitude', 'Amplitude after C', 'Amplitude after U_C'],
               [[bits, initial[int(bits, 2)], observed[int(bits, 2)], phased[int(bits, 2)]] for bits in ("0001", "1000")])
    show_table(['Comparison', 'Squared vector norm'], [['Original', float(np.vdot(initial, initial).real)],
                                                ['After C', float(np.vdot(observed, observed).real)],
                                                ['After U_C', float(np.vdot(phased, phased).real)]])
    print('Phase observation used gamma=', gamma, '. Write the state expression for general gamma separately.')
    return {"diagonal": diagonal, "gamma": gamma, "initial": initial,
            "cost_applied": observed, "unitary_applied": phased}

if RUN_OBSERVATIONS:
    _ = run_demo("2a", experiment_2a, (CHECK_LABELS[0],))
else:
    LAB_RESULTS.pop("2a", None)
    print('[Waiting] Predict the score and phase actions in 2(a) first.')

### (b) Implementing Edge-Based Cost Operations

Write the following function.

```python
cost_layer(gamma, cost_scale=1.0)
```

`cost_scale` is a positive real multiplier for the full score. Use its default value 1.0 in the current experiment. Including this parameter lets the same tool handle the change of score units in the final application.

Writing the scale as $s$, the required operation is

$$
U_C(\gamma;s)
=
e^{-i\gamma s\widehat C}
$$

The same scale applies to every edge score, while the graph's connectivity stays unchanged.

First compare the single-edge operation

$$
e^{-i\gamma s(I-Z_iZ_j)/2}
$$

with $R_{ZZ}(\lambda)$ and find:

- The required rotation angle $\lambda$
- The global phase remaining between the two expressions
- Why the order of the edge operations may be changed

Use the result to implement the function.

The function must satisfy the following conditions.

- Return a `QuantumCircuit(4)` with four qubits and no classical bits.
- Include the cost operations for all five edges in `EDGES`.
- Use $R_{ZZ}$ or its equivalent CX–$R_z$–CX decomposition.
- The full circuit handles initial H gates and the mixer. This function returns only the cost operation.
- Do not include measurement, reset, or initialize.
- Handle general real $\gamma$ and positive `cost_scale` values.
- A global phase common to the entire operation is allowed.

A cost layer acts on an already prepared state. Correct behavior on $|0000\rangle$ alone is not enough. At multiple angles and scales, the check cell tests <strong>whether the full cost operation matches its definition</strong>.

---

In [ ]:
def cost_layer(gamma, cost_scale=1.0):
    circuit = QuantumCircuit(4)
    # TODO BEGIN — Implement the cost operation for the score and scale.
    raise NotImplementedError('cost_layer')
    # TODO END
    return circuit

In [ ]:
_ = check_cost_layer(cost_layer)

### Answer 2b

Calculations, results, and reasoning:

…

### (c) Writing the $p=1$ QAOA Circuit

Write the following function.

```python
qaoa_p1(gamma, beta, cost_scale=1.0)
```

Return a circuit that starts from $|0000\rangle$ and prepares

$$
|\psi(\gamma,\beta;s)\rangle
=
U_B(\beta)U_C(\gamma;s)|+\rangle^{\otimes4}
$$

Use $s=1$ in the base experiment.

First fix

$$
e^{-i\beta X}
\quad\text{and}\quad
R_x(\lambda)=e^{-i\lambda X/2}
$$

Compare these expressions to find the mixer rotation angle $\lambda$.

Then connect these three parts in order:

1. Prepare the uniform superposition on four qubits.
2. Use the circuit returned by your `cost_layer`.
3. Apply the mixer on all four qubits.

The function must satisfy the following conditions.

- Return a `QuantumCircuit(4)` with four qubits and no classical bits.
- Use the supplied `gamma`, `beta`, and `cost_scale`.
- Call your `cost_layer` and connect its returned circuit for the cost part.
- Use H, $R_{ZZ}$, and $R_x$. The equivalent CX–$R_z$–CX cost decomposition is also allowed.
- Do not use measurement, reset, initialize, or extra qubits.
- Prepare the defined state at general input angles, rather than a circuit valid only for a chosen optimal bitstring or particular angles.

After writing the circuit, explain the following.

- Which initial distribution do the H gates prepare?
- Which angles did you pass to cost and mixer, and how did you determine their coefficients and signs?
- How did you translate the operator order $U_BU_C$ into code order?
- Which part uses `cost_scale` when it changes?

The check cell compares the full joint state with the defined state for several inputs. Global phases and equivalent gate implementations are allowed, but <strong>circuits are not considered equivalent merely because a few probabilities or the final score happen to match</strong>.

In [ ]:
def qaoa_p1(gamma, beta, cost_scale=1.0):
    circuit = QuantumCircuit(4)
    # TODO BEGIN — Connect initial preparation, your cost_layer, and the mixer.
    raise NotImplementedError('qaoa_p1')
    # TODO END
    return circuit

In [ ]:
_ = check_qaoa_p1(qaoa_p1)

### Answer 2c

Calculations, results, and reasoning:

…

## Problem 3. Do the Two Operations Actually Change the Output Distribution?

You have connected score-dependent cost phases to a mixer that can convert phase differences into output amplitudes. Now <strong>vary the angles to increase the mean score and investigate each operation's role</strong>.

Keep the graph and edge scores unchanged. Fix `cost_scale` at 1.0 and adjust only cost angle $\gamma$ and mixer angle $\beta$.

First identify the value passed to the classical optimizer. Then optimize and compare circuits at the same angles with the mixer removed or the operation order reversed.

---

### The Goal Is to Increase the Mean Score

Write the state prepared at $(\gamma,\beta)$ as

$$
|\psi(\gamma,\beta)\rangle
$$

If $P_{\gamma,\beta}(z)$ is the probability of obtaining bitstring $z$ in a Z measurement, its mean score is

$$
\mu(\gamma,\beta)
=
\sum_z P_{\gamma,\beta}(z)C(z)
$$



Using the score operator from the previous problem, the same value is

$$
\mu(\gamma,\beta)
=
\langle\psi(\gamma,\beta)|
\widehat C
|\psi(\gamma,\beta)\rangle
$$



Here, $\mu$ is the <strong>mean cut score as a function of the angles</strong>. Assigning more probability to high-scoring bitstrings can raise the mean.

Every bitstring scores at least 0 and at most $C_{\max}$, so the mean of a normalized distribution also satisfies

$$
0\le\mu(\gamma,\beta)\le C_{\max}
$$

Thus, the maximum found in Problem 1 is also a reference for interpreting mean scores.

---

### Passing a Maximization Goal to a Minimizer

The provided classical optimizer searches for <strong>smaller objective values</strong>, whereas you want to increase the mean score $\mu$.

Connect these goals by defining

$$
f(\gamma,\beta)=-\mu(\gamma,\beta)
$$



For example, candidate mean scores 2.4 and 2.9 correspond to objective values $-2.4$ and $-2.9$. The minimizer prefers the smaller value $-2.9$, corresponding to the higher mean score 2.9.

Distinguish these three signs:

| Expression | Role |
|---|---|
| $\widehat C$ | Represents partition scores; a higher value means a better answer. |
| $U_C(\gamma)=e^{-i\gamma\widehat C}$ | Defines the cost circuit used in this lab. |
| $f(\gamma,\beta)=-\langle\widehat C\rangle$ | The value passed to the minimizer to maximize mean score. |

The objective's minus sign determines <strong>how the classical optimizer ranks results</strong>. The cost operation and rotation angles still follow Problem 2's definitions. Do not independently reverse the cost sign just because the objective returns a negative value.

---

### How the Provided Objective Uses Your Circuit

The objective and its connection to the optimizer are provided. Examine what the code calculates, then run it using your circuit from the previous problem.

The provided tool for mean score is

```python
mean_cut_from_circuit(circuit, cost_scale=1.0)
```

It calculates the state prepared by the supplied circuit from $|0000\rangle$ and returns the score operator's exact expectation value. Use `cost_scale=1.0` here.

The objective passed to the optimizer is:

```python
def qaoa_objective(parameters):
    gamma, beta = parameters
    circuit = qaoa_p1(gamma, beta, cost_scale=1.0)
    mean_cut = mean_cut_from_circuit(circuit, cost_scale=1.0)
    return -mean_cut
```

`qaoa_p1` is your function, and it uses your `cost_layer`. Every time the optimizer proposes angles, it therefore evaluates the output of the circuit you actually wrote.

This objective is not a new TODO. Here, examine the relationship between input angles, prepared state, mean score, and return value.

Optimization uses exact expectation values in this part. The next problem examines fluctuations in finite-shot measurement records.

---

### Run the Provided Objective Definition

Run this cell too. It defines the objective explained above and is not a new function TODO.

In [ ]:
def qaoa_objective(parameters):
    gamma, beta = parameters
    circuit = qaoa_p1(gamma, beta, cost_scale=1.0)
    mean_cut = mean_cut_from_circuit(circuit, cost_scale=1.0)
    return -mean_cut

### Each Angle Update Evaluates a Freshly Prepared State

The provided optimization experiment starts from

$$
(\gamma,\beta)=(0.1,0.1)
$$

with angles in radians.

The optimizer passes different angles to the objective and compares returned values to choose the next candidate. Every evaluation starts fresh from $|0000\rangle$. It does not append the next candidate's gates to the state prepared for the previous candidate.

The execution record contains:

| Record | Meaning |
|---|---|
| Input angles | $\gamma,\beta$ used for that evaluation |
| Mean score | Exact $\langle\widehat C\rangle$ of the circuit's state |
| Objective value | $-\langle\widehat C\rangle$ returned to the optimizer |
| Evaluation count | Number of objective calls |
| Termination information | Status and reason for stopping the search |

The optimizer may try angles with lower mean scores while comparing nearby candidates, so the recorded mean need not increase at every evaluation. One optimization iteration may call the objective multiple times; distinguish evaluation count from iteration count.

After optimization, prepare the circuit again at the returned angles and check the mean. This is <strong>the result found in this run</strong>, not proof of a maximum over every possible angle.

---

### Changing Operations to Examine Their Roles

After observing a change in mean score, investigate which operations produced it.

The standard circuit is

$$
U_B(\beta)U_C(\gamma)|+\rangle^{\otimes4}
$$

Fix the angles and compare the following cases.

| Case | Operations |
|---|---|
| Initial state | Measure $\lvert+\rangle^{\otimes4}$ directly |
| Standard circuit | Cost followed by mixer |
| Cost only | Apply only cost to the initial state |
| Reversed circuit | Mixer followed by cost |

Reversing operation order generally changes the state. However, do not conclude that every observable must change simply because the order changed. First examine <strong>how the first operation acts on the given initial state</strong>.

For example, when mixer acts first, use

$$
X|+\rangle=|+\rangle
$$

Recall how Problem 2 applied exponential operators to eigenstates, and consider what this relation tells you about the mixer.

The comparison circuits are provided. No new answer function is needed; interpret results with the same graph and angles while changing only the presence or order of operations.

---

### The Same Z Distribution Does Not Necessarily Mean the Same Quantum State

Cost adds phases to computational-basis components. Relative phases can therefore change even when the Z distribution remains the same.

Check two quantities separately in this comparison.

First, compare the output distribution with the initial uniform distribution. The provided tool calculates

$$
D_Z
=
\max_z\left|P(z)-\frac1{16}\right|
$$

This is the largest deviation from the initial probability $1/16$ among the 16 bitstrings. If $D_Z=0$, the Z distribution is still uniform.

Second, compare fidelity with the initial quantum state.

$$
F_{\mathrm{initial}}
=
\left|
\langle+|^{\otimes4}
|\psi_{\mathrm{out}}\rangle
\right|^2.
$$

If $F_{\mathrm{initial}}=1$, the output agrees with the initial state up to a global phase. This quantity checks <strong>whether the initial state was preserved</strong>. It is not the probability of an optimal bitstring or a measure of optimization performance.

Together, $D_Z$ and $F_{\mathrm{initial}}$ can reveal a relative-phase change even when the measurement distribution agrees. Allow small numerical errors using the provided tolerance.

---

### Your Turn

### (a) Checking the Objective Sign and Direction of Evaluation

Use the following two candidate angle pairs.

$$
(\gamma_P,\beta_P)=(0.1,0.1),
\qquad
(\gamma_Q,\beta_Q)=(0.3,0.2).
$$

Calculate their exact mean scores with the provided tool and complete the table.

| Candidate | $(\gamma,\beta)$ | Mean score $\mu$ | Objective value $f$ |
|---|---|---:|---:|
| P | $(0.1,0.1)$ | … | … |
| Q | $(0.3,0.2)$ | … | … |

Use the results to explain the following.

1. Which candidate has the higher mean score? Is it the one preferred by the minimizer because of its smaller objective value?
2. If the objective returned the mean score without a minus sign, which direction would the minimizer favor?
3. Why is adding a minus sign to maximize mean score different from changing the cost circuit's definition?

Finally, check that each candidate's mean score lies within

$$
0\le\mu\le C_{\max}
$$

and derive the corresponding range of objective values.

---

### Answer 3a

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Results: 3a

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_3a():
    rows, results = [], []
    for label, parameters in (("P", (0.1, 0.1)), ("Q", (0.3, 0.2))):
        circuit = qaoa_p1(*parameters, cost_scale=1.0)
        mean = mean_cut_from_circuit(circuit, cost_scale=1.0)
        objective = qaoa_objective(parameters)
        rows.append([label, parameters, mean, objective])
        results.append({"label": label, "parameters": parameters, "mean_cut": mean, "objective_value": objective})
    show_table(['Candidate', "gamma,beta", 'Exact mean score', 'Objective value'], rows)
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("3a", experiment_3a, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("3a", None)
    print('[Waiting] Check the maximization/minimization direction in 3(a), then run.')

### (b) Optimizing Angles and Comparing with the Starting Point

After passing the circuit checks, run the provided optimization experiment from initial angles $(0.1,0.1)$.

Compare these three cases using the same mean-score metric:

- The uniform distribution calculated in Problem 1
- The standard QAOA circuit at initial angles $(0.1,0.1)$
- The standard QAOA circuit at the angles returned by the optimizer

| Case | Angles used | Exact mean score | $C_{\max}-\langle C\rangle$ |
|---|---|---:|---:|
| Uniform distribution | Not applicable | … | … |
| Circuit at initial angles | $(0.1,0.1)$ | … | … |
| Circuit after optimization | … | … | … |

Also record the optimizer's outputs and execution information.

| Item | Result |
|---|---|
| Returned $\gamma,\beta$ | … |
| Returned objective value | … |
| Mean score recalculated at the returned angles | … |
| Number of objective evaluations | … |
| Number of optimization iterations | … |
| Termination message | … |

Then explain the following.

- Did the mean score improve over the initial angles? How does it compare with the uniform distribution?
- Do the returned objective and recalculated mean agree with the required sign relationship?
- If the mean is below $C_{\max}$, can the distribution assign probability only to optimal solutions?
- Does this optimization result alone establish that you found the best of all angles for this $p=1$ circuit?

If some evaluation scores decreased, inspect those records too. Distinguish trying multiple candidates during optimization from the performance of the finally returned angles.

---

### Answer 3b

Calculations, results, and reasoning:

…

### Checking the Results: 3b

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_3b():
    report = optimize_qaoa(qaoa_objective, initial=(0.1, 0.1))
    circuit = qaoa_p1(*report["parameters"], cost_scale=1.0)
    report["recalculated_mean"] = mean_cut_from_circuit(circuit, cost_scale=1.0)
    report["recalculated_objective"] = qaoa_objective(report["parameters"])
    assert np.isclose(report["recalculated_mean"], report["mean_cut"], atol=ATOL, rtol=0), 'Recheck the mean at the returned angles.'
    table, maximum, optimal = classical_reference(score_function=cut_score)
    report["maximum"] = maximum
    report["uniform_mean"] = sum(table.values()) / len(table)
    show_table(['Case', 'Angles', 'Mean score', 'Cmax - mean'],
               [['Uniform distribution', 'Not applicable', report["uniform_mean"], maximum-report["uniform_mean"]],
                ['Initial circuit', report["initial"], report["initial_mean"], maximum-report["initial_mean"]],
                ["Circuit at optimizer's returned angles", report["parameters"], report["recalculated_mean"], maximum-report["recalculated_mean"]]])
    show_table(['Returned objective value', 'Evaluations', 'Number of repetitions', 'Successful termination'],
               [[report["objective_value"], report["nfev"], report["nit"], report["success"]]])
    print('Termination message:', report["message"])
    print("First and last evaluation records (full history: LAB_RESULTS['3b']['history']):")
    show_table(['Evaluation order', "gamma,beta", 'Mean score', 'Objective value'],
               [[index+1, report["history"][index]["parameters"], report["history"][index]["mean_cut"],
                 report["history"][index]["objective_value"]] for index in (0, len(report["history"])-1)])
    print('This is the result found in this run, not proof of a global maximum over all angles.')
    return report

if RUN_OBSERVATIONS:
    _ = run_demo("3b", experiment_3b, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("3b", None)
    print('[Waiting] 3(b): pass the circuit checks, then enable observations.')

### (c) What Happens without the Mixer or with Reversed Order?

Let the angles from (b) be $(\gamma_*,\beta_*)$. Fix both values and compare four cases.

| Case | Output state definition |
|---|---|
| Initial state | $\lvert+\rangle^{\otimes4}$ |
| Standard circuit | $U_B(\beta_*)U_C(\gamma_*)\lvert+\rangle^{\otimes4}$ |
| Cost only | $U_C(\gamma_*)\lvert+\rangle^{\otimes4}$ |
| Reversed circuit | $U_C(\gamma_*)U_B(\beta_*)\lvert+\rangle^{\otimes4}$ |

Before running, predict whether each Z distribution stays uniform and how the mean scores compare. Then run the provided comparison and complete the table.

| Case | Prediction and reasoning before running | $D_Z$ | Exact mean score | $F_{\mathrm{initial}}$ |
|---|---|---:|---:|---:|
| Initial state | … | … | … | … |
| Standard circuit | … | … | … | … |
| Cost only | … | … | … | … |
| Reversed circuit | … | … | … | … |

Explain the following points.

<strong>1. Cost only</strong>

Write the phase multiplying each initial bitstring amplitude $1/4$. Use its squared magnitude to explain the Z distribution, and decide whether the same distribution means the same initial quantum state.

Check whether this circuit is equivalent to setting only $\beta$ to 0 in standard QAOA, while keeping $\gamma$ at $\gamma_*$.

<strong>2. Mixer first</strong>

Derive the mixer's action on the initial state with every qubit in $|+\rangle$. You may use the following two relations.

$$
X|+\rangle=|+\rangle,
\qquad
U_B(\beta)=\prod_{i=0}^{3}e^{-i\beta X_i}.
$$

Compare the state after mixer followed by cost with the cost-only state. Explain the observed distributions and mean scores.

<strong>3. Summarizing the two roles</strong>

Explain what cost and mixer each do in the standard circuit. Connect <strong>score-dependent relative phase to the subsequent combination of amplitudes</strong>, rather than saying “more gates made it better.”

If a circuit preserved the initial output distribution, use both $F_{\mathrm{initial}}$ and the state expression to assess the claim that it is equivalent to doing nothing.

### Answer 3c

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Results: 3c

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_3c():
    if "3b" not in LAB_RESULTS:
        raise ValueError('Run the 3(b) optimization first.')
    gamma, beta = LAB_RESULTS["3b"]["parameters"]
    cost = circuit_unitary(cost_layer(gamma, cost_scale=1.0), "cost")
    mixer = expm(-1j * beta * MIXER_OPERATOR.to_matrix())
    after_cost = Statevector(np.einsum("ij,j->i", cost, INITIAL_STATE.data))
    after_mixer = np.einsum("ij,j->i", mixer, INITIAL_STATE.data)
    reversed_state = Statevector(np.einsum("ij,j->i", cost, after_mixer))
    states = [('Initial state', INITIAL_STATE),
              ('Standard circuit', circuit_state(qaoa_p1(gamma, beta), "qaoa")),
              ('Cost only', after_cost), ('Mixer followed by cost', reversed_state)]
    rows, results = [], []
    for label, state in states:
        probabilities = state.probabilities()
        dz = float(np.max(np.abs(probabilities - 1/16)))
        mean = float(np.real(state.expectation_value(cost_operator())))
        fidelity = float(state_fidelity(INITIAL_STATE, state))
        results.append({"label": label, "state": state, "probabilities": probabilities,
                        "D_Z": dz, "mean_cut": mean, "fidelity_to_initial": fidelity})
        rows.append([label, dz, mean, fidelity])
    show_table(['Case', "D_Z", 'Exact mean score', 'F with initial state'], rows)
    show_table(['Bitstring', *[label for label, state in states]],
               [[bits, *[float(state.probabilities()[i]) for label, state in states]] for i, bits in enumerate(BITSTRINGS)])
    beta_zero = circuit_state(qaoa_p1(gamma, 0.0), "qaoa")
    print('Fidelity between cost-only state and beta=0 circuit:', float(state_fidelity(after_cost, beta_zero)))
    print('F is referenced to the initial |+>^4 state. Distinguish it from optimal-solution probability.')
    return {"parameters": (gamma, beta), "rows": results}

if RUN_OBSERVATIONS:
    _ = run_demo("3c", experiment_3c, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("3c", None)
    print('[Waiting] Predict the removed-operation and reversed-order results in 3(c) first.')

## Problem 4. Which Output Distribution Is Better?

You have found angles that improve the mean and examined cost and mixer. Now investigate <strong>which criteria to use when evaluating the output distribution</strong>.

A high mean score is different from obtaining optimal solutions frequently. Observing one optimal result in finitely many runs also does not determine the circuit's optimal-solution probability.

Write a function that calculates mean score and optimal-solution probability separately, then apply the same evaluation rules to exact distributions and measurement records.

---

### Mean Score Includes Every Answer's Score

For output bitstring $z$ with probability $P(z)$ and score $C(z)$, the mean score is

$$
\langle C\rangle
=
\sum_z P(z)C(z)
$$



Nonoptimal bitstrings also contribute. Frequently producing several high-scoring answers can give a high mean without producing only optimal answers.

The optimizer minimized the negative of this mean. Thus, <strong>mean score was the metric it directly sought to improve</strong>. Calculate other metrics separately to see whether they improved too.

---

### Optimal-Solution Probability Uses the Whole Optimal Set

Let $C_{\max}$ be the maximum from Problem 1 and define the optimal set as

$$
\mathcal Z_{\mathrm{opt}}
=
\{z:C(z)=C_{\max}\}
$$



The probability of an optimal solution is

$$
P_{\mathrm{opt}}
=
\sum_{z\in\mathcal Z_{\mathrm{opt}}}P(z)
$$

This sums the probabilities of every maximum-scoring bitstring.

Distinguish these three quantities:

| Quantity | Meaning |
|---|---|
| Mean score $\langle C\rangle$ | Average score over repeated executions |
| Optimal-solution probability $P_{\mathrm{opt}}$ | Probability of obtaining an optimal answer in one execution |
| Highest observed score | Largest score in the finite record actually obtained |

The probability of the most frequent individual bitstring is not necessarily $P_{\mathrm{opt}}$. Check whether it is optimal, and include the probabilities of all other optimal bitstrings.

Swapping the two group labels preserves the score. The two corresponding strings are separate measurement outcomes, but if both are optimal, both probabilities count as success.

---

### Mean Score and Optimal-Solution Probability Can Rank Circuits Differently

Before comparing two output distributions, specify the goal of the comparison.

If scores from every run are accumulated, mean score is relevant. If a maximum-scoring answer is required, optimal-solution probability is relevant.

One distribution may frequently produce near-optimal answers. Another may produce optimal answers more often but also produce more low-scoring results. The metrics can therefore rank them differently.

Compare two runs with the same graph and $p=1$ circuit, changing only the angles. The goal is to check <strong>whether selecting a different performance metric can change the comparison even when the circuit structure is unchanged</strong>.

---

### Handling Probabilities and Counts with One Rule

Exact state calculations give a probability for each bitstring. Measurement simulations give counts: how many times each bitstring occurred.

To handle both with one function, interpret the input as <strong>weights</strong> assigned to bitstrings. Let $w_z$ be the weight of $z$ and define the total weight as

$$
W=\sum_z w_z
$$

The normalized distribution is then

$$
P_w(z)=\frac{w_z}{W}
$$



For exact probabilities, usually $W=1$. For counts, $W$ is the total number of measurements.

For example, consider this record.

```python
{"0011": 6, "0000": 2}
```

There are 8 recorded outcomes, so the normalized weights are $6/8$ and $2/8$. Using Problem 1's scores, the mean is

$$
\frac{6C(0011)+2C(0000)}8
$$



For optimal-solution probability, also divide by the total $W$, but add only <strong>weights of bitstrings in the full problem's optimal set</strong> to the numerator.

Do not redefine $C_{\max}$ as the highest score among recorded bitstrings: a finite record may contain no true optimum. Use all candidates enumerated in Problem 1 to define the maximum and optimal set.

Treat a missing dictionary key as a bitstring with weight 0.

---

### Sample Values Must Remain within the Score Range

With fixed angles, suppose $N$ circuit runs produce

$$
z^{(1)},z^{(2)},\ldots,z^{(N)}
$$

The sample mean score is

$$
\widehat{\langle C\rangle}
=
\frac1N\sum_{k=1}^{N}C\bigl(z^{(k)}\bigr)
$$



Records can vary for the same circuit, so this may differ from the exact mean. The observed optimal-solution fraction can also differ from the exact $P_{\mathrm{opt}}$.

However, every measured bitstring's score satisfies

$$
0\le C\bigl(z^{(k)}\bigr)\le C_{\max}
$$

so its sample average must also satisfy

$$
0\le\widehat{\langle C\rangle}\le C_{\max}
$$

The observed optimal-solution fraction likewise lies between 0 and 1.

Lab 4 combined Pauli expectation estimates from different measurement settings into an energy. Here, <strong>each Z measurement gives a valid bitstring, whose score is calculated and averaged</strong>. A sample mean above the possible maximum therefore cannot be explained solely by finite-shot fluctuations. Check scoring, normalization, and interpretation of the records.

---

### Comparing under the Same Conditions

Keep the graph and scoring definition the same when comparing circuits. For sample results, also record the shot count.

The base experiment keeps `cost_scale=1.0` and uses 1,024 shots per circuit. After choosing the angles, do not optimize again in response to the sampled results. This experiment compares the exact distribution's performance with a finite record sampled from it.

Exact metrics describe the circuit's distribution; sample metrics estimate them from finite records. Do not require decimal-for-decimal equality. Clearly state which quantities and records you compared.

---

### Your Turn

### (a) Comparing Output Distributions at Two Angle Pairs

Prepare standard $p=1$ QAOA circuits at the following angles.

$$
(\gamma_A,\beta_A)
=
\left(\frac{\pi}{6},\frac{\pi}{16}\right),
$$

$$
(\gamma_B,\beta_B)
=
\left(\frac{\pi}{4},\frac{\pi}{8}\right).
$$

The graph, initial state, circuit structure, and `cost_scale=1.0` are identical. Only the angles differ.

The provided comparison shows exact probabilities for each bitstring. Complete the table using Problem 1's score table and optimal set.

| Run | Mean score $\langle C\rangle$ | Optimal-solution probability $P_{\mathrm{opt}}$ |
|---|---:|---:|
| A | … | … |
| B | … | … |

Calculate the mean by summing score times probability, and optimal-solution probability by summing over all optimal bitstrings. You may also check the mean against `mean_cut_from_circuit` from the previous problem.

After calculating, explain the following.

1. Which run has the higher mean score?
2. Which run has the higher probability of an optimal solution?
3. Which run would you select when accumulating scores from every execution, and which when an optimal answer is required?
4. Which metric did the classical optimizer directly improve? Does improving it alone establish that the other metric also improved?

If the rankings differ, do not immediately assume a calculation is wrong. Explain which bitstrings' information each metric includes.

---

### Answer 4a

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Results: 4a

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_4a():
    score_table, maximum, optimal = classical_reference(score_function=cut_score)
    results = {}
    for label, parameters in (("A", (np.pi/6, np.pi/16)), ("B", (np.pi/4, np.pi/8))):
        circuit = qaoa_p1(*parameters, cost_scale=1.0)
        results[label] = {"parameters": parameters, "probabilities": probabilities_from_circuit(circuit),
                          "operator_mean": mean_cut_from_circuit(circuit)}
    show_table(['Bitstring', 'Student score', 'A probability', 'B probability'],
               [[bits, score_table[bits], results["A"]["probabilities"][bits], results["B"]["probabilities"][bits]] for bits in BITSTRINGS])
    show_table(['Run', "gamma,beta", 'Mean from observable'],
               [[name, row["parameters"], row["operator_mean"]] for name, row in results.items()])
    print('First calculate optimal-solution probability by summing over the full optimal set from Problem 1.')
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4a", experiment_4a, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("4a", None)
    print('[Waiting] 4(a): enable observations when ready to compare the two metrics.')

### (b) Writing a Function for Both Distribution Metrics

Write the following function.

```python
distribution_metrics(weights, cost_scale=1.0)
```

`weights` is a dictionary whose keys are bitstrings and whose values are probabilities or measurement counts.

The input is guaranteed to satisfy:

- Keys are valid four-bit strings in `z3z2z1z0` order.
- Values are finite and nonnegative.
- The total weight is positive.
- Some bitstrings may be absent.

`cost_scale` is a <strong>positive multiplier for all partition scores</strong>, not for the input record weights. For scale $s$, the score is

$$
C_s(z)=sC(z)
$$

Use $s=1$ in the base experiment and another scale in the final application.

Distinguish the two inputs' roles.

| Input | Meaning |
|---|---|
| `weights` | Probability or observed count assigned to each bitstring |
| `cost_scale` | Common multiplier for each partition's score |

Obtain the reference from the provided tool.

```python
score_table, maximum, optimal_strings = classical_reference(cost_scale)
```

For the full set of 16 bitstrings, it returns:

- `score_table`: each bitstring's score $C_s(z)$
- `maximum`: the maximum over all candidates
- `optimal_strings`: every bitstring attaining that maximum

The function must return a tuple containing these two real values:

```python
(mean_cut, p_opt)
```

- `mean_cut`: mean score after normalizing input weights by their total
- `p_opt`: sum of normalized weights for bitstrings in the full problem's optimal set

In particular, check the following:

1. Use the same rule for probabilities and counts.
2. Treat absent bitstrings as having weight 0.
3. Do not sum only one optimal solution's probability.
4. Do not redefine the maximum using only the observed bitstrings.

The check cell tests exact probabilities, integer counts, sparse records, and several score scales.

After completing the function, pass it the two distributions from (a) and check agreement with your hand calculations of both metrics.

---

In [ ]:
def distribution_metrics(weights, cost_scale=1.0):
    # TODO BEGIN — Calculate both metrics using the total weight and exact optimal set.
    raise NotImplementedError('distribution_metrics')
    # TODO END

In [ ]:
_ = check_distribution_metrics(distribution_metrics)

### Answer 4b

Calculations, results, and reasoning:

…

### Checking the Results: 4b

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_4b():
    if "4a" not in LAB_RESULTS:
        raise ValueError('Run the probability table in 4(a) first.')
    rows, results = [], {}
    for name, data in LAB_RESULTS["4a"].items():
        mean, optimal = distribution_metrics(dict(data["probabilities"]), cost_scale=1.0)
        rows.append([name, data["parameters"], mean, optimal])
        results[name] = {"mean_cut": float(mean), "p_opt": float(optimal)}
    show_table(['Run', "gamma,beta", "Student function's mean score", "Student function's optimal-solution probability"], rows)
    print('Compare with your hand calculations in 4(a) and state which metric you used.')
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4b", experiment_4b, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4b", None)
    print('[Waiting] 4(b): check the metrics function, then compare with your 4(a) calculations.')

### (c) Reviewing Finite Measurement Records and Performance Reports

Compare these three cases:

- Uniform distribution
- Circuit at Problem 3's initial angles $(0.1,0.1)$
- Circuit at the angles returned by Problem 3's optimizer

The provided experiment generates 1,024-shot records for each case. Pass both exact probabilities and counts to your `distribution_metrics` and complete the table.

| Case | Exact mean score | Sample mean score | Exact optimal-solution probability | Sample optimal-solution fraction | Highest observed score |
|---|---:|---:|---:|---:|---:|
| Uniform distribution | … | … | … | … | … |
| Circuit at initial angles | … | … | … | … | … |
| Circuit after optimization | … | … | … | … | … |

For the optimized circuit's record, select a most frequent bitstring and record its score and observed fraction. If several tie for the largest count, you may choose any one.

Examine the reports below. Explain what each establishes and which conclusions are premature.

> <strong>Report A</strong>  
> “An optimal solution was observed from this circuit, so it performs better than uniform bitstring sampling.”

Distinguish the highest observed score from distribution performance. Explain which metrics should be compared with the uniform baseline.

> <strong>Report B</strong>  
> “A run with a higher mean score also has a higher probability of an optimal solution.”

Evaluate this using the two runs in (a). Explain what information each metric represents.

> <strong>Report C</strong>  
> “I will report the observed fraction of the most frequent bitstring as this circuit's optimal-solution probability.”

Check whether that string is optimal and whether other optimal strings exist. Which records must be combined to calculate the correct sample optimal-solution fraction?

> <strong>Report D</strong>  
> “The sample mean score exceeded $C_{\max}$, but that must be a finite-shot fluctuation.”

Evaluate this by identifying the values being averaged. Also explain how this bitstring-score average differs from Lab 4's energy estimate.

Finally, list what a circuit-performance report should record. Include at least <strong>the graph and scoring definition, angles used, exact versus sample values, shots, and performance metrics compared</strong>.

---

### Answer 4c

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Results: 4c

Write the prediction and calculations for the question before running. Once the required function checks pass, the experiment uses your actual functions and the provided tools. If it says it is waiting, check the experiment execution setting and the earlier check cells.

In [ ]:
def experiment_4c():
    if "3b" not in LAB_RESULTS:
        raise ValueError('Run the 3(b) optimization first.')
    uniform = QuantumCircuit(4)
    uniform.h(range(4))
    circuits = [('Uniform distribution', uniform), ('Initial angle', qaoa_p1(0.1, 0.1)),
                ('After optimization', qaoa_p1(*LAB_RESULTS["3b"]["parameters"]))]
    results, rows = [], []
    for index, (label, circuit) in enumerate(circuits):
        probabilities = probabilities_from_circuit(circuit)
        counts = sample_counts(circuit, shots=1024, seed=20261109+index)
        exact_mean, exact_opt = distribution_metrics(dict(probabilities), cost_scale=1.0)
        sample_mean, sample_opt = distribution_metrics(dict(counts), cost_scale=1.0)
        best = max(cut_score(bits) for bits in counts)
        mode = max(counts, key=lambda bits: (counts[bits], bits))
        results.append({"label": label, "probabilities": probabilities, "counts": counts,
                        "exact_mean": float(exact_mean), "exact_p_opt": float(exact_opt),
                        "sample_mean": float(sample_mean), "sample_p_opt": float(sample_opt),
                        "observed_best": best, "mode": mode, "mode_score": cut_score(mode),
                        "mode_frequency": counts[mode]/1024, "shots": 1024})
        rows.append([label, exact_mean, sample_mean, exact_opt, sample_opt, best])
    show_table(['Case', 'Exact mean', 'Sample mean', 'Exact optimal-solution probability', 'Sample optimal-solution fraction', 'Highest observed score'], rows)
    show_table(['Bitstring', *[row["label"] for row in results]],
               [[bits, *[row["counts"].get(bits, 0) for row in results]] for bits in BITSTRINGS])
    show_table(['Case', 'One most frequent bitstring', 'Its score', 'Observed fraction'],
               [[row["label"], row["mode"], row["mode_score"], row["mode_frequency"]] for row in results])
    print('Every case uses 1,024 shots. Ties for the most frequent result are broken by string order.')
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4c", experiment_4c, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4c", None)
    print('[Waiting] 4(c): compare exact distributions and samples under matching conditions.')

## Final Application — What If Every Edge's Score Doubles?

Keep the graph's connectivity, initial state, and circuit structure unchanged, but change each edge's score from 1 to 2.

$$
C'(z)=2C(z),
\qquad
\widehat C'=2\widehat C.
$$

This changes not only how outputs are scored: <strong>the cost circuit also uses the new score operator</strong>. Set `cost_scale=2.0` in the code.

Let $(\gamma_*,\beta_*)$ be the angles found earlier. Compare keeping those angles with changing them to preserve the original output distribution.

First predict the following.

1. How do the maximum score and optimal-solution set change?
2. If only `cost_scale` changes to 2.0 and the angles stay fixed, are the cost phases the same as before?
3. How should $\gamma,\beta$ be adjusted to preserve the original distribution?

Propose the angle change before running another optimization. Explain it using the definition from Problem 2,

$$
U_C(\gamma;s)=e^{-i\gamma s\widehat C}
$$



Then use the provided comparison tool to obtain exact distributions for three cases.

| Case | `cost_scale` | Angles |
|---|---:|---|
| Original experiment | 1.0 | $(\gamma_*,\beta_*)$ |
| Score scale changed only | 2.0 | $(\gamma_*,\beta_*)$ |
| Score scale and angles changed | 2.0 | Your proposed angles |

When evaluating each distribution, pass its `cost_scale` to `distribution_metrics` as well. The circuit's score definition must agree with the one used to evaluate its output.

Writing the original distribution as $P_{\mathrm{original}}(z)$, the tool also calculates each distribution's difference from it:

$$
D_{\mathrm{original}}
=
\max_z
\left|P(z)-P_{\mathrm{original}}(z)\right|
$$

This comparison uses exact probabilities, not sample counts.

| Case | Mean in the corresponding score units | Maximum for that problem | Optimal-solution probability | $D_{\mathrm{original}}$ |
|---|---:|---:|---:|---:|
| Original experiment | … | … | … | … |
| Score scale changed only | … | … | … | … |
| Score scale and angles changed | … | … | … | … |

After examining the results, explain the following.

- Did your angle change preserve the original output distribution? Do the defining equation and the probability comparison agree?
- Even if the distribution is preserved, how does the numerical mean score change? How do the maximum score and optimal-solution probability compare?
- Is it appropriate to say, “The mean score doubled, so the algorithm's performance improved twofold”?

No new circuit or optimizer is needed. <strong>Distinguish a change of score units from an improvement in the output distribution</strong>, and summarize which parameters you changed to preserve which condition.

## Final Application Answer

Prediction and calculations before running:

…

Results and explanation:

…

### Checking the Application

Enter your proposed `(gamma, beta)` in `APPLICATION_PARAMETERS`. Set `APPLICATION_PREDICTIONS` to a dictionary with three keys: `maximum` (new maximum score), `optimal_strings` (tuple of new optimal bitstrings), and `same_angles_keep_distribution` (whether the same angles preserve the distribution, True or False). After writing your predictions and reasoning, set `RUN_APPLICATION=True` and run the experiment cell. These variables are not new function TODOs.

In [ ]:
RUN_APPLICATION = False
APPLICATION_PARAMETERS = None
APPLICATION_PREDICTIONS = None

In [ ]:
def application_experiment():
    if "3b" not in LAB_RESULTS:
        raise ValueError('Run the 3(b) optimization first.')
    if APPLICATION_PARAMETERS is None or APPLICATION_PREDICTIONS is None:
        raise ValueError('Enter your proposed angles and all three predictions first.')
    proposed = tuple(map(float, APPLICATION_PARAMETERS))
    if len(proposed) != 2 or not np.isfinite(proposed).all():
        raise ValueError('Proposed angles must be two finite values: gamma and beta.')
    predictions = APPLICATION_PREDICTIONS
    if not isinstance(predictions, dict) or not {'maximum', 'optimal_strings', 'same_angles_keep_distribution'} <= set(predictions):
        raise ValueError('Check the prediction keys maximum, optimal_strings, and same_angles_keep_distribution.')
    if not isinstance(predictions['same_angles_keep_distribution'], (bool, np.bool_)):
        raise ValueError('Predict distribution preservation with True or False.')
    baseline = LAB_RESULTS["3b"]["parameters"]
    original = weight_vector(probabilities_from_circuit(qaoa_p1(*baseline, cost_scale=1.0)))
    results, rows = [], []
    for label, parameters, scale in (('Original experiment', baseline, 1.0),
                                     ('Scale changed only', baseline, 2.0),
                                     ('Scale and angles changed', proposed, 2.0)):
        circuit = qaoa_p1(*parameters, cost_scale=scale)
        probabilities = probabilities_from_circuit(circuit)
        mean, optimal_probability = distribution_metrics(dict(probabilities), cost_scale=scale)
        score_table, maximum, optimal_strings = classical_reference(scale, score_function=cut_score)
        difference = float(np.max(np.abs(weight_vector(probabilities)-original)))
        rows.append([label, scale, parameters, mean, maximum, optimal_probability, difference])
        results.append({"label": label, "cost_scale": scale, "parameters": parameters,
                        "probabilities": probabilities, "mean_cut": float(mean),
                        "maximum": maximum, "optimal_strings": optimal_strings,
                        "p_opt": float(optimal_probability), "distribution_difference": difference})
    show_table(['Case', 'Scale', 'Angles', 'Mean in corresponding units', 'Maximum score', 'Optimal-solution probability', 'Difference from original distribution'], rows)
    same_angles_preserve = results[1]["distribution_difference"] <= ATOL
    checks = {"maximum": bool(np.isclose(float(predictions['maximum']), results[1]['maximum'], atol=ATOL, rtol=0)),
              "optimal_strings": set(predictions['optimal_strings']) == set(results[1]['optimal_strings']),
              "same_angles_keep_distribution": bool(predictions['same_angles_keep_distribution']) == same_angles_preserve}
    show_table(['Prediction item', 'Your prediction', 'Actual result', 'Matches'],
               [['New maximum score', predictions['maximum'], results[1]['maximum'], checks['maximum']],
                ['New optimal set', tuple(predictions['optimal_strings']), results[1]['optimal_strings'], checks['optimal_strings']],
                ['Original angles preserve distribution', predictions['same_angles_keep_distribution'], same_angles_preserve, checks['same_angles_keep_distribution']]])
    preserved = results[2]["distribution_difference"] <= ATOL
    print('Proposed angles preserve original distribution:', preserved)
    print('The comparison tests distribution preservation; state fidelity=1 is not a separate success requirement.')
    return {"rows": results, "prediction_checks": checks, "predictions_match": all(checks.values()),
            "distribution_preserved": preserved}

LAB_RESULTS.pop("application", None)
if RUN_APPLICATION:
    _ = run_demo("application", application_experiment, CHECK_LABELS)
else:
    print('[Waiting] Final application: write predictions and proposed angles, then enable the separate execution setting.')

## Lab 5 Wrap-Up

Check all four code-check statuses and the explanations for every subquestion. In three or four sentences, connect the score function to its observable and phase operation, explain the roles of cost and mixer, distinguish mean score from optimal-solution probability, and distinguish changed score units from a better output distribution.

### Wrap-Up Answer

Calculations, results, and reasoning:

…

In [ ]:
expected_checks = ['1b · Partition score', '2b · Cost operation', '2c · QAOA layer', '4b · Distribution metrics']
show_table(['Code check', 'State'], [[name, CHECK_STATUS.get(name, 'Not run')] for name in expected_checks])
if all(CHECK_STATUS.get(name) == 'Passed' for name in expected_checks):
    print('All four code checks passed. Review written answers and the final application separately.')
else:
    print('Complete and run the TODO and check cells, then rerun the connected experiments.')